[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/stabilizer/stabilizer.ipynb)

# Four-Qubit Stabilizer Code

This example builds a four-qubit quantum stabilizer code, the smallest code that detects single-qubit bit flips and phase flips alike while storing two logical qubits. One syndrome check tests whether flipping all four physical qubits leaves the state unchanged, and a second tests whether phase-flipping all four does. Encoded states pass both checks, whereas any error on a single qubit flips at least one check outcome so that the corrupted run is rejected, leaving undetected logical errors only when at least two qubits suffer errors in the same run.

An algebra with four positive-square and four negative-square generators has 256 basis blades, matching the space of real linear operators on sixteen real four-qubit amplitudes, and a quantum state is modelled directly inside the same algebra as its density operator. Every four-qubit pattern of bit and phase flips corresponds up to sign to a single basis blade, so the syndrome checks, single-qubit errors, and logical operators are all blades, and multiplying the commuting factors `(1 + check) / 2` projects onto the code subspace. Conjugating a state by `turn * initial_state * turn.inverse()` evolves or corrupts it, and `n_amplitudes * operator.scalar_product(state)` evaluates any observable expectation, as well as pure-state fidelity when the observable is the target state itself. State preparation and bit- or phase-flip noise stay within real amplitudes, whereas arbitrary four-qubit circuits require sixteen complex amplitudes; a [twin spinor notebook](stabilizer_spinors.ipynb) contrasts the left-ideal spinor representation.

In Pauli stabilizer and Dirac notation, the $[[4, 2, 2]]$ code has stabilizer generators $S_1 = X \otimes X \otimes X \otimes X$ and $S_2 = Z \otimes Z \otimes Z \otimes Z$ with logical operators $\bar X_1 = X \otimes X \otimes I \otimes I$, $\bar Z_1 = Z \otimes I \otimes Z \otimes I$, $\bar X_2 = X \otimes I \otimes X \otimes I$, and $\bar Z_2 = Z \otimes Z \otimes I \otimes I$, while the real two-error operator $X Z = -i Y$ carries the same transition probabilities as $Y$ on real-amplitude states. Evolving a $16 \times 16$ density matrix $\rho = |\psi\rangle\langle\psi|$ under a four-fold Kronecker product $U = \bigotimes_{k=1}^4 U_k$ writes $\rho \mapsto U \rho U^\dagger$ and evaluates expectations via the matrix trace $\operatorname{tr}(O \rho)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from functools import reduce
from operator import mul

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure
from examples.quantum.stabilizer import render

ga = Algebra("a+b+c+d+e-f-g-h-")
mv = NumpyContext(ga).multivector

Scalar = ga.gatype.scalar()                                                    # Scalar+
# Real linear operators on sixteen four-qubit amplitudes, including density-operator states.
Operator = ga.gatype.full()                                                    # Full+
# Multiplying scalar_product by 16 recovers the operator trace over the sixteen real amplitudes.
n_amplitudes = 16
identity = mv.scalar()                                                         # [] Scalar+

# Bit flips and phase flips square to +1, anticommute on the same qubit, and commute across distinct qubits.
flips = stack([mv.b, -mv.bce, -mv.bcdef, -mv.ah])                              # [4] Operator+
phase_flips = stack([mv.be, mv.cf, mv.dg, -mv.abcdefg])                        # [4] Operator+
# Each qubit undergoes the identity, a bit flip, a phase flip, or their product.
errors = stack([identity.broadcast_to(flips.shape), flips, phase_flips, flips * phase_flips], axis=-1)  # [4, 4] Operator+
# Multiplying the four +1 phase-flip projectors prepares the all-zero state.
ground = reduce(mul, (1 + phase_flips) / 2)                                    # [] Operator+

## Stabilizer checks and logical operators

Each stabilizer check multiplies four single-qubit blades, one per physical qubit. Because a qubit's bit flip and phase flip anticommute and the two checks overlap on all four qubits, an even number of anticommutations makes the two checks commute. Multiplying the half-projectors `(1 + checks * outcomes) / 2` across all four sign patterns builds four orthogonal syndrome projectors that sum to the identity, with the `(+1, +1)` projector defining the code subspace. Each logical flip and phase flip acts on a pair of physical qubits, overlapping each check on two qubits so that all logical operators commute with both checks.

In [ ]:
# One check flips all four qubits and the other phase-flips all four.
checks = stack([reduce(mul, flips), reduce(mul, phase_flips)])                 # [2] Operator+
# Pairing both checks with all four sign patterns builds four orthogonal syndrome projectors, with (+1, +1) as the code.
outcomes = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]])                      # [4, 2]
halves = (1 + checks * outcomes) / 2                                           # [4, 2] Operator+
projectors = halves[:, 0] * halves[:, 1]                                       # [4] Operator+
code = projectors[0]                                                           # [] Operator+

# Pairwise bit flips and phase flips commute with both checks and act on the two encoded qubits.
logical_flips = stack([flips[0] * flips[1], flips[0] * flips[2]])              # [2] Operator+
logical_phase_flips = stack([phase_flips[0] * phase_flips[2], phase_flips[0] * phase_flips[1]])  # [2] Operator+
logical_operators = stack([logical_flips, logical_phase_flips], axis=-1)       # [2, 2] Operator+

## Encoding logical superpositions

Projecting the all-zero state into the code prepares the logical zero state, and exponentiating each logical qubit's flip times its phase flip turns that qubit continuously toward 1. Because those generator blades square to `-1` without needing to have grade 2, their reverse need not equal their inverse, and states transform by `turn * initial_state * turn.inverse()`. Every logical turn commutes with both stabilizer checks, so the encoded superposition keeps check expectations equal to `+1` while the logical flip and phase-flip expectations recover the sine and cosine of each angle.

In [ ]:
from examples.quantum.stabilizer.scenarios import angles

In [ ]:
def turned_state(
    initial_state: Operator,
    qubit_flips: Operator,
    qubit_phase_flips: Operator,
    angles: np.ndarray,
) -> Operator:
    """Conjugation by the exponential of each qubit's flip times its phase flip turns `initial_state` from 0 toward 1 by `angles` (`[...] Operator+`)."""
    # Multi-blade generators squaring to -1 need not be grade-2 bivectors, so turn.inverse() need not equal ~turn.
    turns = ((qubit_flips * qubit_phase_flips) * (angles / 2)).exp()           # [..., 2] Operator+
    turn = turns[..., 0] * turns[..., 1]                                       # [...] Operator+
    return turn * initial_state * turn.inverse()                               # [...] Operator+


def encoded_state(angles: np.ndarray) -> Operator:
    """Projecting the four-qubit ground state into the code at unit trace and turning both logical qubits encodes `angles` (`[...] Operator+`)."""
    projected_ground = code * ground * code                                    # [] Operator+
    code_ground = projected_ground / (n_amplitudes * identity.scalar_product(projected_ground))  # [] Operator+
    return turned_state(code_ground, logical_flips, logical_phase_flips, angles)  # [...] Operator+


stored_state = encoded_state(angles)                                           # [] Operator+
check_readings = n_amplitudes * checks.scalar_product(stored_state)            # [2] Scalar+
logical_readings = n_amplitudes * logical_operators.scalar_product(stored_state)  # [2, 2] Scalar+

print("checks:", check_readings.to_array())
print("flip and phase flip of each encoded qubit:\n", logical_readings.to_array())
print("sine and cosine of each angle:\n", np.stack([np.sin(angles), np.cos(angles)], axis=-1))

## Detecting single-qubit errors

A bit flip on any single qubit anticommutes with the four-qubit phase-flip check, a phase flip anticommutes with the bit-flip check, and a combined flip-and-phase error anticommutes with both. Conjugating the encoded density operator by each single-qubit error therefore flips at least one check expectation from `+1` to `-1` in every non-trivial column of the syndrome table. Because all four qubits produce the same syndrome for a given error type, the two checks flag corrupted runs for rejection while two-qubit errors such as a pair of bit flips can still commute with both checks.

In [ ]:
def single_error_readings(state: Operator) -> Scalar:
    """Conjugating `state` by each single-qubit error yields both check expectations across four qubits and four error kinds (`[4, 4, 2] Scalar+`)."""
    corrupted_states = errors * state * errors.inverse()                       # [4, 4] Operator+
    return n_amplitudes * checks.scalar_product(corrupted_states[..., None])   # [4, 4, 2] Scalar+


readings = single_error_readings(stored_state)                                 # [4, 4, 2] Scalar+
print(render.syndrome_table(readings))

## Acceptance and fidelity under noise

Under independent depolarizing noise, each physical qubit either stays untouched or suffers one of the three non-trivial errors with equal probability, producing 256 four-qubit error patterns weighted by their joint probabilities. Because a pure density operator equals its own rank-one projector, `n_amplitudes * stored_state.scalar_product(corrupted_states)` directly measures fidelity with the intended state, while `n_amplitudes * code.scalar_product(corrupted_states)` gives the probability of passing both checks. Rejected runs have zero overlap with the code subspace where `stored_state` lives, so dividing average fidelity by the acceptance probability yields the fidelity conditional on passing both checks.

In matrix notation, the real algebra maps isomorphically to $\mathbb{R}^{16 \times 16}$ by sending each basis blade to a four-fold Kronecker product of the $2 \times 2$ matrices $I$, $X$, $Z$, and $XZ$, so a pure state is a symmetric idempotent matrix $\rho$ of trace $1$ and post-selected fidelity evaluates $\sum_m p_m \operatorname{tr}(\rho\, E_m \rho E_m^{-1}) / \sum_m p_m \operatorname{tr}(P_{\mathrm{code}}\, E_m \rho E_m^{-1})$.

In [ ]:
from examples.quantum.stabilizer.scenarios import n_rates, rates

In [ ]:
def error_patterns(n_qubits: int, rates: np.ndarray) -> tuple[Operator, np.ndarray]:
    """Independent depolarizing noise on `n_qubits` qubits pairs every multi-qubit error blade (`[...] Operator+`) with its probability (`[...]`) across `rates`."""
    kinds = np.indices((errors.shape[-1],) * n_qubits)                         # [n_qubits, ...]
    chances = np.stack([1 - rates, rates / 3, rates / 3, rates / 3], axis=-1)  # [..., 4]
    patterns = reduce(mul, (errors[qubit, kind] for qubit, kind in enumerate(kinds)))  # [...] Operator+
    weights = reduce(mul, (chances[:, kind] for kind in kinds))                # [...]
    return patterns, weights


def pattern_average(values: Scalar, weights: np.ndarray) -> Scalar:
    """Weighting `values` by `weights` sums over the trailing error-pattern axes to give the mean at each noise rate (`[...] Scalar+`)."""
    return (values * weights).sum(axis=tuple(range(1 - weights.ndim, 0)))      # [...] Scalar+

In [ ]:
def code_performance(stored_state: Operator, rates: np.ndarray) -> tuple[Scalar, Scalar]:
    """Depolarizing noise on all four physical qubits gives the fraction of runs passing both checks and the conditional infidelity of accepted runs (`[...] Scalar+`)."""
    patterns, weights = error_patterns(errors.shape[0], rates)
    corrupted_states = patterns * stored_state * patterns.inverse()            # [4, 4, 4, 4] Operator+
    # Expecting the code projector gives the probability that both checks read +1.
    acceptance = n_amplitudes * code.scalar_product(corrupted_states)          # [4, 4, 4, 4] Scalar+
    # Because a pure density operator equals its own projector, its expectation in the corrupted state equals the fidelity.
    fidelity = n_amplitudes * stored_state.scalar_product(corrupted_states)    # [4, 4, 4, 4] Scalar+
    kept_fraction = pattern_average(acceptance, weights)                       # [...] Scalar+
    return kept_fraction, 1 - pattern_average(fidelity, weights) / kept_fraction  # [...] Scalar+, Scalar+


def bare_infidelity(angles: np.ndarray, rates: np.ndarray) -> Scalar:
    """Depolarizing noise on two unencoded qubits at `angles` gives their unconditional infidelity at each rate (`[...] Scalar+`)."""
    n_bare = angles.shape[-1]
    bare_state = turned_state(ground, flips[:n_bare], phase_flips[:n_bare], angles)  # [] Operator+
    patterns, weights = error_patterns(n_bare, rates)
    corrupted_states = patterns * bare_state * patterns.inverse()              # [4, 4] Operator+
    fidelity = n_amplitudes * bare_state.scalar_product(corrupted_states)      # [4, 4] Scalar+
    return 1 - pattern_average(fidelity, weights)                              # [...] Scalar+


kept_fraction, encoded_errors = code_performance(stored_state, rates)          # [n_rates] Scalar+, Scalar+
bare_errors = bare_infidelity(angles, rates)                                   # [n_rates] Scalar+

save_figure(render.noise_figure(rates, kept_fraction, encoded_errors, bare_errors), "stabilizer_noise");

The blue curve shows the conditional infidelity of accepted encoded runs, which grows quadratically at small per-qubit error rates because every single-qubit error is rejected, whereas the red curve for two unencoded qubits grows linearly from the origin. The dashed grey curve tracks the fraction of runs passing both stabilizer checks, falling from `1` as single-qubit rejections become frequent.

In [ ]:
# checks
assert (projectors.sum(axis=0) - 1).scalar_norm_squared() < 1e-20
assert ((check_readings - 1).abs() < 1e-12).all()
expected_logical = np.stack([np.sin(angles), np.cos(angles)], axis=-1)
assert ((logical_readings - expected_logical).abs() < 1e-12).all()
assert ((readings[:, 0] - 1).abs() < 1e-12).all()
assert (readings[:, 1:] < 0).any(axis=-1).all()
# Doubling a small error rate quadruples conditional infidelity and doubles bare-qubit infidelity.
assert (encoded_errors[2] / encoded_errors[1] - 4).abs() < 0.2
assert (bare_errors[2] / bare_errors[1] - 2).abs() < 0.1